# Caso: reducción de la deserción escolar

## Artefactos Cognitivos: Taxonomía, Ontología y Epistemología

Este Colab presenta una versión deliberadamente pequeña de tres artefactos cognitivos construidos a partir del caso del **Programa de Alerta Temprana para prevenir la deserción escolar**.

La intención no es representar exhaustivamente el sistema educativo. El objetivo es **ver cómo se materializa cada artefacto** y poder manipularlo.

| Artefacto | Pregunta |
|---|---|
| **Taxonomía** | ¿Qué elementos y tipos reconocemos? |
| **Ontología** | ¿Cómo se relacionan esos elementos? |
| **Epistemología** | ¿Qué necesitamos conocer y bajo qué criterios consideraremos que comprendemos o resolvemos el problema? |

> **Importante:** estas son aproximaciones iniciales. No existe una única taxonomía u ontología posible para el caso.


## 1. El caso, muy brevemente

Una región implementa un programa para identificar tempranamente a estudiantes con riesgo de abandonar sus estudios. Una norma establece que todo estudiante de alto riesgo debe recibir acompañamiento.

Se construye un modelo de *machine learning* con registros administrativos. El modelo tiene buen desempeño predictivo (**Accuracy 91 %, AUC 0,93**) y utiliza, entre otros, asistencia, rendimiento, repitencia, cambios de institución y distancia al colegio.

El sistema identifica **10 000 estudiantes** de alto riesgo, pero la región dispone de unos **500 orientadores** y existen otras restricciones: diferencias de infraestructura, horarios familiares, transporte, trabajo o cuidado de terceros, registros desiguales y factores no observados.

La pregunta que queda abierta es:

**¿Qué necesitamos representar y conocer además del riesgo predictivo para que la política pueda funcionar en la realidad?**


## 2. Taxonomía

La taxonomía organiza los **tipos de elementos** que reconocemos en este mundo del problema.

Aquí utilizaremos cinco dimensiones, cada una con una definición y cuatro tipos.


In [ ]:
taxonomia = {
    "Actores": {
        "definicion": "Entidades que participan, deciden o reciben efectos dentro del sistema.",
        "tipos": ["Estudiante", "Familia", "Docente u orientador", "Autoridad educativa"]
    },
    "Condiciones": {
        "definicion": "Circunstancias que condicionan las posibilidades de funcionamiento del sistema.",
        "tipos": ["Infraestructura", "Recursos humanos", "Condiciones familiares", "Transporte y acceso"]
    },
    "Factores educativos": {
        "definicion": "Elementos asociados con la trayectoria educativa del estudiante.",
        "tipos": ["Asistencia", "Rendimiento", "Material educativo", "Repitencia"]
    },
    "Intervenciones": {
        "definicion": "Acciones mediante las cuales el sistema intenta modificar una situación.",
        "tipos": ["Alerta", "Acompañamiento", "Derivación", "Apoyo educativo"]
    },
    "Resultados": {
        "definicion": "Estados o resultados que pueden utilizarse para evaluar la trayectoria del sistema.",
        "tipos": ["Persistencia", "Aprendizaje", "Satisfacción académica", "Competencias laborales"]
    }
}

taxonomia


### Una taxonomía también puede representarse como JSON

Esto permite guardar el artefacto, intercambiarlo o procesarlo posteriormente con código.


In [ ]:
import json

print(json.dumps(taxonomia, ensure_ascii=False, indent=2))


## 3. Ontología

Ahora dejamos de preguntar solamente **qué elementos reconocemos** y empezamos a explicitar **cómo se relacionan**.

Para mantener el ejemplo manejable, utilizaremos una red pequeña. Cada relación tiene tres partes:

**origen → relación → destino**


In [ ]:
ontologia = [
    {"origen": "Infraestructura", "relacion": "condiciona", "destino": "Capacidad de aprendizaje"},
    {"origen": "Material educativo", "relacion": "favorece", "destino": "Capacidad de aprendizaje"},
    {"origen": "Recursos humanos", "relacion": "hacen posible", "destino": "Acompañamiento"},
    {"origen": "Condiciones familiares", "relacion": "condicionan", "destino": "Asistencia"},
    {"origen": "Asistencia", "relacion": "influye en", "destino": "Capacidad de aprendizaje"},
    {"origen": "Capacidad de aprendizaje", "relacion": "influye en", "destino": "Satisfacción académica"},
    {"origen": "Satisfacción académica", "relacion": "influye en", "destino": "Persistencia"},
    {"origen": "Acompañamiento", "relacion": "puede favorecer", "destino": "Persistencia"},
    {"origen": "Transporte y acceso", "relacion": "condiciona", "destino": "Asistencia"},
    {"origen": "Estudiante", "relacion": "recibe", "destino": "Acompañamiento"}
]

ontologia


### La misma ontología como tabla

La tabla es útil porque permite revisar y editar las relaciones con facilidad.


In [ ]:
import pandas as pd

ontologia_df = pd.DataFrame(ontologia)
ontologia_df


## 4. De la tabla de relaciones a una red NetworkX

La siguiente función convierte una tabla con las columnas `origen`, `relacion` y `destino` en una red de **NetworkX**.

La idea es que no sea necesario aprender NetworkX para comenzar a experimentar con redes. El estudiante solamente necesita construir o modificar el artefacto conceptual; la función se ocupa de convertirlo en una visualización.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def ontologia_a_red(df):
    """Convierte una tabla de relaciones en un grafo dirigido de NetworkX.

    La tabla debe contener:
      origen | relacion | destino
    """
    G = nx.DiGraph()

    for _, fila in df.iterrows():
        origen = fila["origen"]
        relacion = fila["relacion"]
        destino = fila["destino"]
        G.add_edge(origen, destino, relacion=relacion)

    return G


def visualizar_ontologia(df, figsize=(13, 9), seed=7):
    """Construye y muestra una visualización sencilla de la ontología."""
    G = ontologia_a_red(df)

    plt.figure(figsize=figsize)
    pos = nx.spring_layout(G, seed=seed, k=1.8)

    nx.draw_networkx_nodes(G, pos, node_size=2300, alpha=0.85)
    nx.draw_networkx_edges(
        G, pos,
        arrows=True,
        arrowsize=18,
        width=1.5,
        connectionstyle="arc3,rad=0.05"
    )
    nx.draw_networkx_labels(G, pos, font_size=10)

    etiquetas = nx.get_edge_attributes(G, "relacion")
    nx.draw_networkx_edge_labels(
        G, pos,
        edge_labels=etiquetas,
        font_size=9,
        label_pos=0.5
    )

    plt.title("Ontología del caso: red de relaciones")
    plt.axis("off")
    plt.show()

    return G


### Ejecutamos la función

Observa que aquí no necesitamos escribir código de NetworkX. La función recibe nuestro artefacto y se ocupa de generar la red.


In [ ]:
G = visualizar_ontologia(ontologia_df)


## 5. Una pequeña exploración de la red

Aunque no necesitamos aprender NetworkX para construir la ontología, podemos observar algunas propiedades de la red.


In [ ]:
print("Número de nodos:", G.number_of_nodes())
print("Número de relaciones:", G.number_of_edges())

print("\nNodos:")
print(list(G.nodes))

print("\nRelaciones:")
for origen, destino, datos in G.edges(data=True):
    print(f"{origen} --[{datos['relacion']}]--> {destino}")


### Una observación importante

El modelo predictivo del caso podría concentrarse en relaciones como:

**Asistencia → riesgo de abandono**

La ontología construida aquí abre el problema y permite preguntar por relaciones que son relevantes para la intervención:

**Condiciones familiares → Asistencia → Capacidad de aprendizaje → Satisfacción → Persistencia**

También aparece una restricción institucional:

**Recursos humanos → Acompañamiento → Persistencia**

La red no pretende demostrar que todas estas relaciones sean causales. Su función en este ejercicio es **hacer explícita una hipótesis sobre cómo funciona el mundo del problema**.


## 6. Epistemología

En este ejemplo la epistemología no se presenta como otra red. Se presenta como una pequeña narrativa que explicita **qué consideramos relevante conocer y qué significa que el sistema esté funcionando**.


In [ ]:
epistemologia = {
    "pregunta_central": "¿Qué significa que el sistema educativo esté funcionando?",
    "posibles_criterios_de_exito": [
        "Aumentar la persistencia",
        "Mejorar la calidad del aprendizaje",
        "Aumentar la satisfacción académica",
        "Desarrollar competencias laborales"
    ],
    "consideraciones": [
        "Persistir no implica necesariamente aprender mejor.",
        "Predecir abandono no demuestra que podamos evitarlo.",
        "Una relación predictiva no necesariamente explica el mecanismo.",
        "La capacidad de intervención condiciona el valor práctico de una predicción."
    ]
}

print(json.dumps(epistemologia, ensure_ascii=False, indent=2))


### Narrativa epistemológica mínima

Un sistema educativo puede considerar que su principal éxito consiste en aumentar la **persistencia** de los estudiantes, pero esto no garantiza necesariamente un mejor **aprendizaje**. También puede priorizar la calidad del aprendizaje, aun cuando ello no maximice el número de estudiantes que permanecen en el sistema. Otra perspectiva puede considerar como resultado principal el desarrollo de **competencias relevantes para la vida y el trabajo**. Por ello, persistencia, aprendizaje, satisfacción y competencias no son necesariamente equivalentes ni pueden utilizarse indistintamente como evidencia de éxito. El modelo y las intervenciones deberían evaluarse de acuerdo con el propósito que el sistema educativo haya decidido privilegiar y con las condiciones reales bajo las cuales ese propósito puede alcanzarse.


## 7. Los tres artefactos juntos

| | Taxonomía | Ontología | Epistemología |
|---|---|---|---|
| **Pregunta** | ¿Qué reconocemos? | ¿Cómo se relaciona? | ¿Qué necesitamos conocer y considerar válido? |
| **Forma en este ejemplo** | JSON jerárquico | Tabla + red | Narrativa + criterios |
| **Función** | Ordenar elementos | Explicitar relaciones | Explicitar criterios, supuestos y propósito |

Los tres artefactos se construyen sobre el mismo caso, pero **no representan lo mismo**.


## 8. ¿Qué ocurre si cambia nuestra forma de entender el problema?

Supongamos que la autoridad decide que el objetivo principal ya no es simplemente **maximizar la persistencia**, sino mejorar la **calidad del aprendizaje**.

Entonces podemos volver a mirar nuestros artefactos:

- ¿La taxonomía contiene los elementos que necesitamos?
- ¿La ontología contiene las relaciones relevantes para ese propósito?
- ¿Necesitamos agregar otros resultados o condiciones?
- ¿Cambiarían las intervenciones?
- ¿Qué información necesitaríamos para evaluar si estamos avanzando?

La idea es mostrar que los artefactos cognitivos **pueden evolucionar junto con nuestra comprensión del problema**.


# 9. Ahora tu propio caso

Para el siguiente ejercicio, construye una primera versión de los tres artefactos para tu propio caso.

### Taxonomía
- 5 dimensiones o categorías relevantes.
- 1 definición breve para cada una.
- Hasta 4 tipos por categoría.

### Ontología
- Hasta 10 nodos inicialmente.
- Relaciones expresadas como **origen → relación → destino**.
- Puedes comenzar con una tabla como la utilizada aquí.
- Luego utiliza la función `visualizar_ontologia()` para observar la red.

### Epistemología
- Una pregunta central.
- Hasta 4 posibles criterios de éxito o resultados relevantes.
- Hasta 4 consideraciones sobre qué necesitamos conocer, distinguir o comprobar.

> **No busques completitud. Construye una primera representación que pueda ser discutida, modificada y mejorada.**


## 10. Una última pregunta

**¿Qué cosas de tu caso se vuelven visibles cuando pasas del modelo o descripción inicial a estos tres artefactos?**

Ese cambio de representación es parte del trabajo de Ingeniería Cognitiva.
